# Tarea 7 · Resolución de versiones

Notebook de solo lectura: no crea tablas, no modifica Bronze y no ejecuta `MERGE`. Requiere que los lotes de reenvío y corrección ya estén cargados y transformados en Silver.


In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(root / 'scripts'))
from deduplicar_mediciones import KEY, resolver_versiones
config = json.loads((root / 'config' / 'silver.json').read_text(encoding='utf-8'))
mediciones = spark.table(config['tables']['mediciones'])
candidatas, reenvios, conflictos, superadas = resolver_versiones(mediciones)
assert candidatas.count() == candidatas.select(*KEY).distinct().count()
print('Silver:', mediciones.count(), 'candidatas:', candidatas.count(), 'reenvíos:', reenvios.count(), 'filas en conflicto:', conflictos.count())
display(candidatas.select(*KEY, 'version_origen', 'reproducciones', '_source_file', 'resolution'))


Silver: 469 candidatas: 467 reenvíos: 1 filas en conflicto: 0


,plataforma,video_id,fecha_medicion,version_origen,reproducciones,_source_file,resolution
0,instagram,V000,2026-11-01,1,88,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv,ORIGINAL
1,instagram,V000,2026-11-02,1,152,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_02/mediciones_2026-11-02.csv,ORIGINAL
2,instagram,V000,2026-11-03,1,222,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv,ORIGINAL
3,instagram,V000,2026-11-04,1,261,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv,ORIGINAL
4,instagram,V000,2026-11-05,1,272,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv,ORIGINAL
5,instagram,V000,2026-11-06,1,407,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv,ORIGINAL
6,instagram,V000,2026-11-07,1,471,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-07.csv,ORIGINAL
7,instagram,V000,2026-11-08,1,630,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv,ORIGINAL
8,instagram,V000,2026-11-09,1,762,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-09.csv,ORIGINAL
9,instagram,V000,2026-11-10,1,877,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-10.csv,ORIGINAL


## Reenvío y corrección reales

La prueba falla explícitamente si alguno de los dos lotes aún no figura en Silver.


In [ ]:
key_filter = ((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000') &
              (F.col('fecha_medicion') == F.lit('2026-11-01').cast('date')))
sample = mediciones.filter(key_filter)
expected_files = ('mediciones_2026-11-01.csv', 'reenvio_dia_1_v1.csv', 'correccion_dia_1.csv')
for filename in expected_files:
    assert sample.filter(F.col('_source_file').endswith(filename)).count() == 1, f'Falta {filename} en Silver'
chosen = candidatas.filter(key_filter).collect()
assert len(chosen) == 1 and chosen[0]['version_origen'] == 2 and chosen[0]['reproducciones'] == 950
assert chosen[0]['_source_file'].endswith('correccion_dia_1.csv')
assert reenvios.filter(key_filter).count() == 1
print('Reenvío idéntico identificado; corrección v2 seleccionada con su procedencia.')
display(sample.select(*KEY, 'version_origen', 'reproducciones', '_source_file'))


## Versión antigua tardía y conflicto

Se construyen dos casos adicionales en memoria. Ninguno escribe datos. La versión antigua recibida después de la nueva no la sustituye. Un conflicto en la versión máxima bloquea la candidatura de esa clave.


In [ ]:
rows = [r.asDict() for r in sample.collect()]
old = next(r for r in rows if r['version_origen'] == 1).copy()
old['_source_file'] = old['_source_file'] + '.late'
late_batch = spark.createDataFrame(rows + [old], schema=sample.schema)
late_candidates, late_reenvios, _, late_superadas = resolver_versiones(late_batch)
assert late_candidates.count() == 1
assert late_candidates.first()['version_origen'] == 2
assert late_reenvios.count() == 2
assert late_superadas.count() == 3

correction = next(r for r in rows if r['version_origen'] == 2).copy()
correction['reproducciones'] += 1
correction['_source_file'] += '.conflict'
conflict_batch = spark.createDataFrame(rows + [correction], schema=sample.schema)
blocked, _, conflict_rows, _ = resolver_versiones(conflict_batch)
assert blocked.count() == 0 and conflict_rows.count() == 2
assert set(r['conflict_code'] for r in conflict_rows.collect()) == {'SAME_KEY_VERSION_DIFFERENT_METRICS'}
print('Versión tardía ignorada; conflicto v2 registrado y clave bloqueada.')
display(conflict_rows.select(*KEY, 'version_origen', 'reproducciones', '_source_file', 'conflict_code'))
